# LLM with Key Value Caching


# Implementing Transformer Architecture: A Step-by-Step Guide


- Key sections:
- 3.1: Encoder and Decoder Stacks
- 3.2: Attention Mechanism
- 3.3: Position-wise Feed-Forward Networks
- 3.4: Embeddings and Softmax
- 3.5: Positional Encoding
- 5.4: Regularization (dropout strategy)

## Implementation Strategy
Breaking down the architecture into manageable pieces and gradually adding complexity:

1. Start with foundational components:
    - Embedding + Positional Encoding
    - Single-head self-attention

2. Build up attention mechanism:
- Extend to multi-head attention
- Add cross-attention capability
- Implement attention masking

3. Construct larger components:
- Encoder (self-attention + FFN)
- Decoder (masked self-attention + cross-attention + FFN)

4. Combine into final architecture:
- Encoder-Decoder stack
- Full Transformer with input/output layers

## Development Tips
1. Visualization and Planning:
- Draw out tensor dimensions on paper
- Sketch attention patterns and masks
- Map each component back to paper equations
- This helps catch dimension mismatches early!

2. Dimension Cheat Sheet:
- Input tokens: [batch_size, seq_len]
- Embeddings: [batch_size, seq_len, d_model]
- Attention matrices: [batch_size, num_heads, seq_len, seq_len]
- FFN hidden layer: [batch_size, seq_len, d_ff]
- Output logits: [batch_size, seq_len, vocab_size]

3. Common Pitfalls:
- Forgetting to scale dot products by √d_k
- Applying mask too early or too late
- Incorrect mask dimensions or application
- Missing residual connections
- Wrong order of layer norm and dropout
- Tensor dimension mismatches in attention
- Not handling padding properly

4. Performance Considerations:
- Memory usage scales with sequence length squared
- Attention computation is O(n²) with sequence length
- Balance between d_model and num_heads
- Trade-off between model size and batch size

## Testing Strategy
- Test each component independently
- Verify shape preservation
- Check attention patterns
- Confirm mask effectiveness
- Validate gradient flow
- Monitor numerical stability

Remember: The key to successfully implementing the Transformer is understanding how each piece fits together and maintaining clear dimension tracking throughout the implementation

In [1]:
import os

import pandas as pd

# Sets the high watermark ratio to 0.0, completely disabling the upper memory allocation limits
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

In [2]:
checkpoint_dir = "checkpoints/"

In [3]:
import gc
import torch


def clean_memory_cache():

    if torch.mps.is_available():
        print(
            f"Before Clearing, Available memory: {torch.mps.driver_allocated_memory() / 1024 / 1024:.2f} MB"
        )
        gc.collect()
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        print(
            f"Before Clearing, Available memory: {torch.cuda.memory_allocated() / 1024 / 1024:.2f} MB"
        )
        gc.collect()
        torch.cuda.empty_cache()
    else:
        gc.collect()
        return

## Transformer and Vision Transformer

In [4]:
from typing import Optional, Tuple, Any, List
import torch
import torch.nn as nn
from dataclasses import dataclass

import numpy as np
import torch
import torchvision
from torch import nn, optim
from torchvision import transforms
import sklearn
from sklearn.metrics import confusion_matrix
import tqdm
import copy
from torch.utils.data import DataLoader, Subset

torch.autograd.set_detect_anomaly(True)
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
import os

num_workers = min(2, os.cpu_count())
import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="vit.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


result = TrainResult(train_losses=[], train_accs=[], val_accs=[], val_losses=[])


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0.0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0.0

    def calculate(self) -> float:
        return self.avg


def print_variance(name: str, data: torch.Tensor):
    # Compute variance across features/neurons and average across the batch
    neuron_variance = torch.mean(torch.var(data.detach().float(), dim=-1))
    print(f"{name}: Variance = {neuron_variance.item():.6f}")


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.scale = n_hidden**-0.5

        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape

        # Shape: (B, T, 3 * num_heads * n_hidden) -> (B, num_heads, T, 3 * n_hidden)
        qkv = (
            self.qkv_projection(x)
            .reshape(B, T, self.num_heads, 3 * self.n_hidden)
            .transpose(1, 2)
        )
        # todo implement caching here for llm

        q, k, v = qkv.chunk(3, dim=-1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if attn_mask is not None:
            if attn_mask.dim() == 3:
                attn_mask = attn_mask.unsqueeze(1)
            scores = scores.masked_fill(attn_mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)

        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        output = self.W0(context)
        return output, attn_weights


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)

        # LayerNorm applied inside the FFN sequence only
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Attention block with residual connection (no norm)
        attn_out, alphas = self.attn(x, attn_mask=attn_mask)
        x = x + attn_out

        # FFN block with residual connection (norm is first layer inside self.ffn)
        x = x + self.ffn(x)
        return x, alphas


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        return_attn: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        collected_attns = []

        for layer in self.layers:
            x, alphas = layer(x, attn_mask=attn_mask)
            if return_attn:
                collected_attns.append(alphas)

        if return_attn:
            return x, torch.stack(collected_attns, dim=1)
        return x, None


class PatchEmbed(nn.Module):
    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert (
            img_size % patch_size == 0
        ), "Image dimensions must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Output: (B, num_patches, nout) in float
        return self.proj(x).flatten(2).transpose(1, 2)


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
        num_global_tokens: int = 1,  # Number of global/CLS tokens
    ):
        super().__init__()
        self.num_global_tokens = num_global_tokens

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        # Learnable global/CLS tokens of shape (1, num_global_tokens, dim)

        self.cls_tokens = nn.Parameter(torch.zeros(1, num_global_tokens, dim))

        # Position embeddings covering both global tokens and image patches
        total_seq_len = num_global_tokens + num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, total_seq_len, dim))

        nn.init.trunc_normal_(self.cls_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification / Projection Head
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # 1. Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # 2. Expand global tokens across batch: (B, num_global_tokens, dim)
        cls_tokens = self.cls_tokens.expand(B, -1, -1)

        # 3. Concatenate global tokens with patch embeddings: (B, num_global_tokens + num_patches, dim)
        x = torch.cat((cls_tokens, embs), dim=1)

        # 4. Add position embeddings
        x = x + self.pos_embed

        # 5. Transformer forward pass
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)

        # 6. Extract representations of all global tokens: (B, num_global_tokens, dim)
        global_repr = x[:, : self.num_global_tokens]

        # Aggregate global tokens:
        # - Option A: Use the primary token (index 0) if acting as a single CLS token with register tokens
        # out = self.head(global_repr[:, 0])
        #
        # - Option B: Mean-pool across all global tokens
        out = self.head(global_repr.mean(dim=1))  # (B, nout)

        return out, alphas


# evaluate the model
@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = AverageMeter(), AverageMeter()
        for img, labels in val_loader:
            # move all img, labels to device (cuda)
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return loss_meter.calculate(), acc_meter.calculate()


result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])


def main():
    model = VisionTransformer(
        n_channels=3,
        nout=10,
        img_size=32,
        patch_size=4,
        dim=128,
        attn_dim=64,
        mlp_dim=128,
        num_heads=3,
        num_layers=6,
    ).to(device)
    # print(model)
    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)]
    )
    inv_transform = transforms.Compose(
        [
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
            transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
            transforms.ToPILImage(),
        ]
    )
    train_dataset = torchvision.datasets.CIFAR10(
        train=True, root="data", transform=img_transform, download=True
    )
    val_dataset = torchvision.datasets.CIFAR10(
        train=False, root="data", transform=img_transform
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    criterion = nn.CrossEntropyLoss()
    NUM_EPOCHS = 10
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-05
    )
    # result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])
    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(
            train_dataloader, desc="Training at " + str(epoch)
        ):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()
        scheduler.step()
        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(
            f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
        )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
        best_models = "Vision_transformer_Best_" + str(epoch + 1) + ".pt"
        best_save_path = os.path.join(checkpoint_dir, best_models)
        model_path = "Vision_transformer_" + str(epoch + 1) + ".pt"
        save_path = os.path.join(checkpoint_dir, model_path)

        # todo add an early stopping criteria and restore best weights

        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:
            torch.save(model.state_dict(), best_save_path)
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
            print("Finished Training")

        else:
            torch.save(model.state_dict(), save_path)
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        clean_memory_cache()
        print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
    print("Finished Training")

In [10]:
device

'mps'

## Testing

In [12]:
num_tokens = 100
batch_size = 10
dim = 64
num_layers = 4
num_heads = 2
dummy_model = Transformer(
    dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
).to(device)

inp = torch.randn(batch_size, num_tokens, dim).to(device)
print(f"Input: {inp.shape=}")
# test case 1 regular forward pass
print("Test Case 1")
with torch.no_grad():
    output, alpha = dummy_model(inp, attn_mask=None)
    # print(f"\n\n\n{output.shape=},{alpha.shape=}")

    # if len(alpha)==0:
    #     alpha=None
    assert alpha is None
    assert output.shape == (
        batch_size,
        num_tokens,
        dim,
    ), f"wrong output shape {output.shape}"

# test case 2 collect attentions

Input: inp.shape=torch.Size([10, 100, 64])
Test Case 1


## Profiling

In [30]:
import torch
from torch.profiler import profile, record_function, ProfilerActivity

# Initialize your custom block and a dummy input tensor

dummy_input = torch.randn(16, 512, 768).to(device)  # [batch, seq_len, d_model]

# Profile execution
with profile(
    activities=[ProfilerActivity.CPU], record_shapes=True, with_stack=True
) as prof:
    with record_function("transformer_forward"):
        # torch.mps.synchronize()
        output = dummy_model(inp)
        # torch.mps.synchronize()
raw_data = prof.key_averages(group_by_input_shape=True)

data_list = []

data_list = []
for item in raw_data:
    # Safely extract shapes (fall back to "N/A" if empty)
    shapes = item.input_shapes if item.input_shapes else "N/A"

    data_list.append(
        {
            "Layer / Operation": item.key,
            "Input Shapes": str(shapes),  # Will now show e.g. "[[16, 512, 768]]"
            "Calls": item.count,
            "Self CPU Time (ms)": item.self_cpu_time_total / 1000.0,
        }
    )

df = pd.DataFrame(data_list)

# 3. Create the Pandas DataFrame
df1 = df.sort_values(by="Self CPU Time (ms)", ascending=False).reset_index(drop=True)

USDT:2026-09-15 22:15:26 3177:50201 SyncActivityProfilerHandler.cpp:52] profiler_start
USDT:2026-09-15 22:15:27 3177:50201 SyncActivityProfilerHandler.cpp:59] profiler_stop


In [31]:
df1[["Layer / Operation", "Input Shapes", "Self CPU Time (ms)"]].head(10)

,Layer / Operation,Input Shapes,Self CPU Time (ms)
0,zmq/sugar/__init__.py(21): device,N/A,78.002711
1,<built-in function read>,N/A,76.979701
2,<built-in method control of select.kqueue obje...,N/A,76.601617
3,traceback.py(528): format_frame_summary,N/A,12.650599
4,traceback.py(459): _extract_from_extended_fram...,N/A,8.826245
5,textwrap.py(416): dedent,N/A,5.318784
6,<built-in function stat>,N/A,4.711183
7,traceback.py(344): _set_lines,N/A,3.641705
8,traceback.py(745): format,N/A,2.771075
9,aten::bmm,"[[20, 100, 32], [20, 32, 100]]",2.497091


In [28]:
df

,Layer / Operation,Calls,CPU Time (ms),Self CPU Time (ms),Input Shapes
0,threading.py(1044): _bootstrap,9,706.608865,0.005083,
1,threading.py(1082): _bootstrap_inner,9,706.603782,0.016585,
2,tqdm/_monitor.py(69): run,1,78.546258,0.000375,
3,threading.py(670): wait,2,157.069373,0.006749,
4,threading.py(373): wait,1,78.539342,0.004375,
...,...,...,...,...,...
303,enum.py(1299): __hash__,1,0.001000,0.000917,
304,<built-in function hash>,1,0.000083,0.000083,
305,torch/profiler/profiler.py(388): stop_trace,1,0.009603,0.004375,
306,torch/autograd/profiler.py(419): __exit__,1,0.005228,0.001959,


In [48]:
import pandas as pd
import torch
from torch.profiler import profile, record_function, ProfilerActivity

# 1. Run the profiler with stack tracking enabled
with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,
    with_stack=True,  # Captures file names and line numbers
    profile_memory=False,  # Keeps profiling overhead light
) as prof:
    with record_function("transformer_forward"):
        if torch.backends.mps.is_available():
            torch.mps.synchronize()

        output = dummy_model(inp)

        if torch.backends.mps.is_available():
            torch.mps.synchronize()

# 2. CRITICAL CHANGE: Group by stack trace to separate sub-calls
raw_data = prof.key_averages(group_by_input_shape=True, group_by_stack_n=5)
system_noise = [
    "ipykernel",
    "threading",
    "asyncio",
    "tornado",
    "IPython",
    "zmq",
    "selectors",
    "tqdm",
    "traceback",
]

data_list = []
for item in raw_data:
    # Skip the massive overarching wrapper to see what's actually taking time inside
    if any(noise in item.key for noise in system_noise):
        continue
    if item.key == "transformer_forward":
        continue

    shapes = item.input_shapes if item.input_shapes else "N/A"

    # Extract the source code location if available
    source_location = (
        item.stack[0] if (hasattr(item, "stack") and item.stack) else "Internal/Core"
    )

    data_list.append(
        {
            "Operation": item.key,
            "Input Shapes": str(shapes),
            "Self CPU Time (ms)": item.self_cpu_time_total / 1000.0,
            "Total CPU Time (ms)": item.cpu_time_total / 1000.0,
            "Calls": item.count,
            "Source File": source_location,
        }
    )

df = pd.DataFrame(data_list)

# 3. Sort by "Self CPU Time" (the work done strictly inside that specific operation)
df = df.sort_values(by="Self CPU Time (ms)", ascending=False).reset_index(drop=True)

# Display the granular breakdown

USDT:2026-09-15 23:39:26 3177:50201 SyncActivityProfilerHandler.cpp:52] profiler_start
USDT:2026-09-15 23:39:26 3177:50201 SyncActivityProfilerHandler.cpp:59] profiler_stop


In [49]:
df[["Operation", "Input Shapes", "Self CPU Time (ms)", "Source File"]]

,Operation,Input Shapes,Self CPU Time (ms),Source File
0,<built-in function read>,N/A,84.645984,Internal/Core
1,<built-in method control of select.kqueue obje...,N/A,84.333099,Internal/Core
2,<built-in method acquire of _thread.lock objec...,N/A,77.452569,Internal/Core
3,<built-in function sleep>,N/A,27.014160,Internal/Core
4,<built-in function _mps_deviceSynchronize>,N/A,7.761121,Internal/Core
...,...,...,...,...
260,<string>(2): __init__,N/A,0.000125,Internal/Core
261,<built-in method append of collections.deque o...,N/A,0.000125,Internal/Core
262,<built-in function allocate_lock>,N/A,0.000125,Internal/Core
263,<built-in method release of _thread.lock objec...,N/A,0.000083,Internal/Core


In [44]:
type(raw_data)

torch.autograd.profiler_util.EventList

In [45]:
for item in raw_data:
    print(item)

<FunctionEventAvg key=threading.py(1044): _bootstrap self_cpu_time=13.558us cpu_time=95.349ms  self_cuda_time=0.000us cuda_time=0.000us input_shapes=[] cpu_memory_usage=0 cuda_memory_usage=0>
<FunctionEventAvg key=threading.py(1082): _bootstrap_inner self_cpu_time=33.542us cpu_time=95.348ms  self_cuda_time=0.000us cuda_time=0.000us input_shapes=[] cpu_memory_usage=0 cuda_memory_usage=0>
<FunctionEventAvg key=tqdm/_monitor.py(69): run self_cpu_time=1.292us cpu_time=95.431ms  self_cuda_time=0.000us cuda_time=0.000us input_shapes=[] cpu_memory_usage=0 cuda_memory_usage=0>
<FunctionEventAvg key=threading.py(670): wait self_cpu_time=2.684us cpu_time=95.429ms  self_cuda_time=0.000us cuda_time=0.000us input_shapes=[] cpu_memory_usage=0 cuda_memory_usage=0>
<FunctionEventAvg key=threading.py(373): wait self_cpu_time=21.791us cpu_time=95.427ms  self_cuda_time=0.000us cuda_time=0.000us input_shapes=[] cpu_memory_usage=0 cuda_memory_usage=0>
<FunctionEventAvg key=ipykernel/parentpoller.py(62): ru

In [42]:
print(len(df))
df[["Operation", "Input Shapes", "Self CPU Time (ms)", "Source File"]]

372


,Operation,Input Shapes,Self CPU Time (ms),Source File
0,zmq/sugar/__init__.py(21): device,N/A,95.301661,Internal/Core
1,<built-in method acquire of _thread.lock objec...,N/A,93.800592,Internal/Core
2,<built-in method control of select.kqueue obje...,N/A,92.490378,Internal/Core
3,<built-in function read>,N/A,92.466070,Internal/Core
4,traceback.py(528): format_frame_summary,N/A,12.675254,Internal/Core
...,...,...,...,...
367,asyncio/base_events.py(2064): get_debug,N/A,0.000083,Internal/Core
368,<built-in method release of _thread.lock objec...,N/A,0.000083,Internal/Core
369,IPython/core/displayhook.py(117): is_active,N/A,0.000082,Internal/Core
370,torch/jit/__init__.py(129): annotate,N/A,0.000042,Internal/Core


In [23]:
df

,Layer / Operation,Calls,CPU Time (ms),Self CPU Time (ms),Input Shapes
0,transformer_forward,1,28.795216,8.834743,
1,aten::linear,16,4.150989,4.122746,
2,aten::empty,44,0.053802,0.053802,
3,aten::reshape,24,4.715110,0.028967,
4,aten::view,20,0.888818,0.888818,
5,aten::transpose,12,1.307492,1.302243,
6,aten::as_strided,40,0.013186,0.013186,
7,aten::chunk,4,0.476718,0.008018,
8,aten::split,4,0.468700,0.443824,
9,aten::narrow,12,0.024876,0.011378,


In [24]:
df.sort_values(by="Self CPU Time (ms)", ascending=False).reset_index(drop=True)

,Layer / Operation,Calls,CPU Time (ms),Self CPU Time (ms),Input Shapes
0,transformer_forward,1,28.795216,8.834743,
1,aten::linear,16,4.150989,4.122746,
2,aten::_unsafe_view,24,2.910057,2.910057,
3,aten::bmm,8,2.515425,2.515425,
4,aten::clone,16,1.798228,1.690619,
5,aten::expand,16,1.661690,1.658254,
6,aten::transpose,12,1.307492,1.302243,
7,aten::_softmax,4,1.144036,1.144036,
8,aten::native_layer_norm,4,0.973063,0.959522,
9,aten::view,20,0.888818,0.888818,


In [20]:
import torch
from torch.profiler import profile, record_function, ProfilerActivity

# Initialize your custom block and a dummy input tensor

dummy_input = torch.randn(16, 512, 768).to(device)  # [batch, seq_len, d_model]

# Profile execution
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    with record_function("transformer_forward"):
        torch.mps.synchronize()
        output = dummy_model(inp)
        torch.mps.synchronize()

# Print the results sorted by GPU (CUDA) execution time
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
        transformer_forward        31.44%      11.161ms       100.00%      35.498ms      35.498ms             1  
               aten::matmul         0.15%      51.906us        26.89%       9.547ms       1.193ms             8  
               aten::linear        15.53%       5.511ms        15.64%       5.551ms     346.907us            16  
              aten::reshape         0.10%      36.850us        15.01%       5.330ms     222.073us            24  
         aten::_unsafe_view         8.52%       3.024ms         8.52%       3.024ms     125.994us            24  
                  aten::bmm         7.96%       2.826ms         7.96%       2.826ms     

USDT:2026-09-15 22:04:15 3177:50201 SyncActivityProfilerHandler.cpp:52] profiler_start
USDT:2026-09-15 22:04:15 3177:50201 SyncActivityProfilerHandler.cpp:59] profiler_stop


In [16]:
with torch.mps.profiler.profile(mode="interval", wait_until_completed=True):
    output = dummy_model(inp)

In [38]:
output[0].detach()

tensor([[[-0.7938,  2.5232,  0.3369,  ...,  1.9120,  1.2901, -0.1501],
         [-0.1579,  0.8397,  1.1667,  ..., -0.7607, -0.3362,  0.8243],
         [ 1.3098,  1.2618,  0.5661,  ..., -0.2068, -1.6439,  0.6374],
         ...,
         [-2.0452,  0.9097,  1.2100,  ...,  0.1006,  2.5819,  0.1403],
         [-0.4268,  0.7689,  1.1859,  ..., -0.2481,  2.6821,  0.1154],
         [-0.5782, -1.1191, -0.3324,  ..., -1.7648, -1.5355, -2.4178]],

        [[-0.0626, -1.6107, -0.3573,  ..., -0.2250, -1.0655, -1.5647],
         [-0.5654,  1.4599, -0.3682,  ..., -0.8329, -0.2141, -1.8950],
         [ 0.3379,  0.5659, -0.7882,  ...,  0.9192,  1.8930, -0.7123],
         ...,
         [ 2.7061,  0.6044,  0.3608,  ..., -2.2541,  1.9104,  1.0061],
         [ 0.0272,  0.4765, -0.2490,  ...,  0.9564, -0.1903,  2.2080],
         [ 0.6145, -0.9165,  0.9108,  ..., -1.5464,  0.3666, -1.6790]],

        [[-0.3031,  0.3775, -0.8756,  ..., -0.1437, -0.4754, -0.9917],
         [ 1.9548,  1.3486,  1.8122,  ..., -0

In [36]:
output[1]

In [5]:
main()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
  )
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x AttentionResidual(
        (attn): MultiHeadedAttention(
          (qkv_projection): Linear(in_features=128, out_features=576, bias=False)
          (W0): Linear(in_features=192, out_features=128, bias=True)
        )
        (ffn): Sequential(
          (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (1): Linear(in_features=128, out_features=128, bias=True)
          (2): GELU(approximate='none')
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Linear(in_features=128, out_features=10, bias=True)
  )
)


Training at 0:   0%|          | 0/196 [00:00<?, ?it/s]/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Training at 0: 100%|██████████| 196/196 [01:03<00:00,  3.08it/s]


Train Epoch: 0, Loss: 1.7765043626785277, Acc: 0.34102


W0915 19:11:39.933000 3177 torch/_inductor/utils.py:1953] [1/0] Not enough SMs to use max_autotune_gemm mode


Val Epoch: 0, Loss: 1.5383171670913696, Acc: 0.4352


...

Train Epoch: 8, Loss: 0.49954819143295287, Acc: 0.826559999961853


Val Epoch: 8, Loss: 0.8463519853591919, Acc: 0.7184
Before Clearing, Available memory: 1123.28 MB
Val Epoch: 9, Loss: 0.8463519853591919, Acc: 0.7184


Training at 9: 100%|██████████| 196/196 [00:59<00:00,  3.30it/s]

Train Epoch: 9, Loss: 0.4464579672050476, Acc: 0.8484400000190735


Val Epoch: 9, Loss: 0.8366068584442139, Acc: 0.72
Before Clearing, Available memory: 1123.28 MB
Val Epoch: 10, Loss: 0.8366068584442139, Acc: 0.72
Finished Training


In [7]:
import pandas as pd

df = pd.DataFrame.from_dict(
    {
        "train_acc": result.train_accs,
        "train_loss": result.train_losses,
        "val_acc": result.val_accs,
        "val_loss": result.val_losses,
    }
)

In [8]:
df

,train_acc,train_loss,val_acc,val_loss
0,0.34102,1.776504,0.4352,1.538317
1,0.50068,1.375106,0.5288,1.289605
2,0.58222,1.160010,0.5990,1.097871
3,0.64070,1.010786,0.6366,1.022503
4,0.68480,0.885047,0.6686,0.923847
5,0.72626,0.772099,0.6816,0.896877
6,0.76326,0.672912,0.7056,0.848598
7,0.79630,0.577325,0.7054,0.847097
8,0.82656,0.499548,0.7184,0.846352
9,0.84844,0.446458,0.7200,0.836607


## LLM

In [11]:
file = "shakespeare.txt"
with open(file, "r") as f:
    dialogues = f.read()

In [12]:
all_dialogues = dialogues.split("\n\n")

In [13]:
for line in all_dialogues[:10]:
    print(line)

First Citizen:
Before we proceed any further, hear me speak.
All:
Speak, speak.
First Citizen:
You are all resolved rather to die than to famish?
All:
Resolved. resolved.
First Citizen:
First, you know Caius Marcius is chief enemy to the people.
All:
We know't, we know't.
First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?
All:
No more talking on't; let it be done: away, away!
Second Citizen:
One word, good citizens.
First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.


In [8]:
import nltk

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /Users/deven/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [9]:
def tokenize(s):
    return nltk.word_tokenize(s)

## Dialogue LLM

In [14]:
def tokenize(s):
    return nltk.word_tokenize(s)


class MyTokenizer:
    def __init__(self, raw_text: str):
        # raw_text     contains the text from which we will build our vocabulary

        self.start = "<START>"  # token that starts every example
        self.pad = "<PAD>"  # token used to pad examples to the same length
        self.unk = "<UNK>"  # token used if encountering a word not in our vocabulary

        vocab = np.unique(tokenize(raw_text))
        vocab = np.concatenate([np.array([self.start, self.pad, self.unk]), vocab])

        self.vocab = vocab  # array of tokens in order
        self.tok_to_id = {w: i for i, w in enumerate(vocab)}  # mapping of token to ID
        self.id_to_token = {i: w for i, w in enumerate(vocab)}
        self.vocab_size = len(self.vocab)  # size of vocabulary

    def __len__(self):
        return self.vocab_size

    def encode(self, s: str) -> torch.Tensor:
        # s           input string
        #
        # Output
        # id_tensor   a tensor of token ids, starting with the start token.t

        id_tensor = torch.from_numpy(
            np.array(
                [self.tok_to_id[self.start]]
                + [self.tok_to_id[w] for w in tokenize(s) if w in self.tok_to_id],
                dtype=np.int32,
            )
        )

        # TODO: tokenize the input using word_tokenize. Return a tensor  of the token ids, starting with the token id for the start token.
        # ============ ANSWER START ===========
        # encoded_string = tokenize(s)
        # token_ids =
        # token_ids.append(self.tok_to_id[self.start])
        # token_ids.extend(
        #     [self.tok_to_id[w] for w in encoded_string if w in self.tok_to_id.keys()]
        # )
        # id_tensor = np.array(token_ids)

        # id_tensor = torch.from_numpy(id_tensor)
        # ============ ANSWER END =============

        return id_tensor

    def decode(self, toks: torch.Tensor) -> str:
        # toks         a list of token ids
        #
        # Output
        # decoded_str  the token ids decoded back into a string (join with a space)

        # TODO: convert the token ids back to the actual corresponding words.
        # Join the tokens with a space and return the full string
        # ============ ANSWER START ===========
        return " ".join(
            [
                self.id_to_token[int(token)]
                for token in toks
                if token in self.tok_to_id.values()
            ]
        ).rstrip()

        # ============ ANSWER END =============

        # return decoded_str

    def pad_examples(self, tok_list: List[torch.Tensor]) -> torch.Tensor:
        # Pads the tensors to the right with the pad token so that they are the same length.
        #
        # tok_list       a list of tensors containing token ids (maybe of different lengths)
        #
        # Output
        # padded_tokens  shape: (len(tok_list), max length within tok_list)
        return torch.nn.utils.rnn.pad_sequence(
            tok_list, batch_first=True, padding_value=self.tok_to_id[self.pad]
        )


tok = MyTokenizer(dialogues)

In [15]:
len(tok)

14058

In [16]:
# tokenizer test cases
input_string = "KING RICHARD III:\nSay that I did all this for love of her. bluye"
enc = tok.encode(input_string)
print(enc)

# for x in enc:
#     print(x, type(x), x in tok.tok_to_id.values(), x in tok.id_to_token.values())
dec = tok.decode(enc)
print(dec)
# print("<START> KING RICHARD III : Say that I did all this for love of her .")
assert dec == "<START> KING RICHARD III : Say that I did all this for love of her ."

tensor([    0,  1396,  1986,  1284,    18,  2148, 12580,  1279,  5523,  3135,
        12633,  6669,  8559,  9443,  7480,    16], dtype=torch.int32)
<START> KING RICHARD III : Say that I did all this for love of her .


In [17]:
class DialogueDataset:
    def __init__(self, tokenizer: MyTokenizer, lines: List[str], max_N: int):
        # tokenizer    an instance of MyTokenizer
        # lines        a list of strings. each element in an example in the dataset
        # max_N        the maximum number of tokens allowed per example. More than this will be truncated
        self.lines = lines
        self.tokenizer = tokenizer
        self.max_N = max_N

    def __len__(self) -> int:
        return len(self.lines)

    # def __iter__(self):
    #     for line in self.lines:
    #         yield self.tokenizer.encode(line)[: self.max_N]

    def __getitem__(self, idx: int) -> torch.Tensor:
        # returns the example at int encoded by the tokenizer
        # truncates the example if it is more than max_N tokens
        return self.tokenizer.encode(self.lines[idx])[: self.max_N]

    # def __getitems__(self,indices:int):
    #     return [self.__getitem__(idx) for idx in indices]

In [18]:
ds = DialogueDataset(tok, all_dialogues, max_N=200)

In [19]:
def collate_fn(examples: List[torch.Tensor]):
    """
    # examples        a batch of tensors containing token ids (maybe of different lengths)
    # Outputs a dictionary containing
    #   input_ids     a single tensor with all of the examples padded (from the right) to the max
    #                 length within the batch. shape:(B, max length within examples)
    #   input_mask    a tensor indicating which tokens are padding and should be ignored. 0 if padding
    #                 and 1 if not. shape: (B, max length within examples)
    """
    new_input_ids = tok.pad_examples(examples)
    attn_mask = torch.ones(new_input_ids.shape)  # 1s should not be ignored

    # causal attention mask
    attn_mask[new_input_ids == tok.tok_to_id[tok.pad]] = (
        0  # should be ignored if it is a padded
    )
    return {"input_ids": new_input_ids, "input_mask": attn_mask}

In [20]:
ds[1]

tensor([    0,   118,    18,  2324,    14, 11846,    16], dtype=torch.int32)

In [23]:
tokens = [ds[1], ds[2]]

In [24]:
type(tokens)

list

In [29]:
collate_fn(tokens)

{'input_ids': tensor([[    0,   118,    18,  2324,    14, 11846,    16,     1,     1,     1,
              1,     1,     1,     1,     1],
         [    0,   950,   505,    18,  2865,  3322,  3135, 10820, 10561, 12760,
           5525, 12571, 12760,  6328,    20]], dtype=torch.int32),
 'input_mask': tensor([[1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])}

In [30]:
tokens[0]

tensor([    0,   118,    18,  2324,    14, 11846,    16], dtype=torch.int32)

In [31]:
tokens[1]

tensor([    0,   950,   505,    18,  2865,  3322,  3135, 10820, 10561, 12760,
         5525, 12571, 12760,  6328,    20], dtype=torch.int32)

In [9]:
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F

In [33]:
tok

In [61]:
train_dl = DataLoader(ds, batch_size=16, num_workers=0, collate_fn=collate_fn)

In [62]:
BATCH_SIZE = 16
BUFFER_SIZE = 4

In [63]:
class DialogueGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_N: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        # vocab_size       size of the vocabulary
        # max_N            maximum number of tokens allowed to appear in 1 example
        # dim              embedding dimension
        # attn_dim         the hidden dimension of the attention layer
        # mlp_dim          the hidden layer dimension of the FFN
        # num_heads        the number of heads in the attention layer
        # num_layers       the number of attention layers.

        super().__init__()
        """
        • Given the token ids, retrieve the corresponding token embeddings. Add to this a learned positional embedding.
        • Generate a causal attention mask. Remember that for GPT, every token only depends on itself and the tokens before it
        • Pass the embeddings and the attention mask to the transformer and the language model head. Output logits of size (T ×V) where T is the number of tokens and V is the vocabulary size. This step is implemented for you
        """
        # TODO: set up the token embedding and positional embeddings
        #       Hint, use nn.Embedding
        # Already Padded to keep sequence length same
        # Next use Graph Neural Network
        # token_embeddings

        self.token_embeddings = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=dim
        )

        self.pos_embeddings = nn.Embedding(num_embeddings=max_N, embedding_dim=dim)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )
        # Projection Head
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, vocab_size))

    def forward(
        self, input_ids: torch.Tensor, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # input_ids     a batch of input ids (right padded). shape: (B x T)
        # return_attn   whether to return the attention weights
        #
        # Output
        # out           the logit vector (B x T x V)
        # alphas        the attention weights if return_attn is True. Otherwise None shape: (B, num_layers, num_heads, T, T)
        """

        Args:
            input_ids: Batch of input ids right padded shape: (BxT)
            return_attn: whether to return the attention weights
        • Generate a causal attention mask. Remember that for Generative  Pretrained Transformer, every token only depends on itself and the tokens before it
        • Pass the embeddings and the attention mask to the transformer and the language model head. Output logits of size (T ×V) where T is the number of tokens and V is the vocabulary size. This step is implemented for you
        Returns:

        """
        B, T = input_ids.shape
        pos_ids = torch.arange(0, T, dtype=torch.long, device=device).unsqueeze(
            0
        )  # Shape: (1, T)
        embs = self.token_embeddings(input_ids) + self.pos_embeddings(pos_ids)
        # TODO: retrieve the token embeddings for the input_ids.
        #       Add to the token embeddings the positional embeddings.
        #       Store the combined embedding in embs

        # TODO: Create the causal attention mask, which should be of size (B, T, T)
        #       Remember that the causal attention mask is lower triangular (all tokens only
        #       depend on themselves and the tokens before them).   Store the mask in causal_attn_mask
        # Hint: check out torch.tril creates a causal mask where each token can only attend to previous tokens and itself
        causal_attn_mask = (
            torch.tril(torch.ones(T, T)).unsqueeze(0).repeat(B, 1, 1)
        ).to(
            device
        )  # Shape: (B, T, T)
        # ============ ANSWER START ============

        # ============ ANSWER END ==============

        x, alphas = self.transformer(
            embs, attn_mask=causal_attn_mask, return_attn=return_attn
        )
        out = self.head(x)
        return out, alphas

    def generate(self, input_ids, num_tokens):
        # you can assume batch size 1
        # greedy generation
        with torch.no_grad():
            for i in range(num_tokens):
                out, _ = self.forward(input_ids)
                new_token = torch.argmax(out[:, [-1]], -1)
                input_ids = torch.cat([input_ids, new_token], dim=1)
        return input_ids

    def key_value_cached_generation(self, input_ids, num_tokens, cache):
        pass

In [64]:
class DialogueLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss(reduction="none")

    def forward(
        self, logits: torch.Tensor, input_ids: torch.Tensor, inp_mask: torch.Tensor
    ):
        """
        # logits      the logits produced by DialogueGPT. shape: (B x T x V)
        # input_ids   the token ids. shape: (B x T)
        # inp_mask    a 0/1 mask of which tokens are padding tokens and should be ignored. shape: (B x T)

        TODO: Implement the language model loss. For logits[i], we want to supervise the i+1 token_id with the cross entropy loss. We thus will not supervise the start token (input_ids[0]) or use the last logit vector (logits[-1]). Return the average of the losses for each token in the batch, making sure to ignore tokens corresponding to the padding (use inp_mask).
        """
        loss = 0

        # start_token = input_ids[0]
        relevant_logits = logits[:, :-1, :]
        # print(f"{logits.shape=},{relevant_logits.shape=}")
        relevant_tokens = input_ids[:, 1:]
        shift_mask = inp_mask[:, 1:]
        relevant_logits = relevant_logits.permute(0, 2, 1)
        # print(f"{relevant_logits.shape=}")

        loss = self.criterion(relevant_logits, relevant_tokens)
        shift_mask = shift_mask.to(dtype=loss.dtype, device=loss.device)
        loss *= shift_mask

        return torch.sum(loss) / torch.clamp(torch.sum(shift_mask), min=1e-9)

In [ ]:
def sample_next_token(input_tokens, model, tokenizer):
    # Run model to get prediction over next output
    outputs = model(
        input_ids=input_tokens["input_ids"],
        attention_mask=input_tokens["attention_mask"],
    )
    # Find prediction
    prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0, -1]
    # TODO Draw a random token according to the probabilities
    # next_token should be an array with an sole integer in it (as below)
    # Use:  https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html
    # Replace this line
    next_token = [5000]

    # Append token to sentence
    output_tokens = input_tokens
    output_tokens["input_ids"] = torch.cat(
        (output_tokens["input_ids"], torch.tensor([next_token])), dim=1
    )
    output_tokens["attention_mask"] = torch.cat(
        (output_tokens["attention_mask"], torch.tensor([[1]])), dim=1
    )
    output_tokens["last_token_prob"] = prob_over_tokens[next_token]

    return output_tokens

In [65]:
import torch.optim as optim

model = DialogueGPT(
    vocab_size=tok.vocab_size,
    max_N=200,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).to(device)
criterion = DialogueLoss()

NUM_EPOCHS = 80

optimizer = optim.AdamW(
    model.parameters(), lr=0.0001, weight_decay=0
)  # implement in homework
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [66]:
model

DialogueGPT(
  (token_embeddings): Embedding(14058, 128)
  (pos_embeddings): Embedding(200, 128)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x AttentionResidual(
        (attn): MultiHeadedAttention(
          (qkv_projection): Linear(in_features=128, out_features=576, bias=False)
          (W0): Linear(in_features=192, out_features=128, bias=True)
        )
        (ffn): Sequential(
          (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (1): Linear(in_features=128, out_features=128, bias=True)
          (2): GELU(approximate='none')
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Linear(in_features=128, out_features=14058, bias=True)
  )
)

In [67]:
# Ensure optimizer starts completely clean before the epoch loop begins
optimizer.zero_grad()

for epoch in range(NUM_EPOCHS):
    loss_meter = AverageMeter()

    # Wrap your loader securely
    for step, inp_dict in tqdm.tqdm(
        enumerate(train_dl), desc=f"Training at {epoch}", total=len(train_dl)
    ):
        inp_ids, inp_mask = inp_dict["input_ids"], inp_dict["input_mask"]

        inp_ids = inp_ids.to(device).long()
        inp_mask = inp_mask.to(device)

        # 1. Forward Pass
        outputs, _ = model(input_ids=inp_ids)

        # 2. FIX: Scale the loss down by BUFFER_SIZE to normalize gradients
        loss = criterion(outputs, inp_ids, inp_mask)
        scaled_loss = loss / BUFFER_SIZE

        # 3. Backward Pass (Accumulates gradients safely)
        scaled_loss.backward()

        # Track the true unscaled loss in your meter
        loss_meter.update(loss.item(), len(inp_dict["input_ids"]))

        # 4. FIX: Step the optimizer on buffer limit OR at the final step of the dataset
        if (step + 1) % BUFFER_SIZE == 0 or (step + 1) == len(train_dl):
            optimizer.step()
            optimizer.zero_grad()  # Resets the buffer for the next accumulation block

            # OPTIONAL: If your scheduler decays per-step instead of per-epoch,
            # place `scheduler.step()` right here.

    # 5. Step scheduler at the epoch level (if using an epoch-based scheduler)
    scheduler.step()

    # 6. Safe Text Generation Example (Switched to eval/inference mode to protect memory buffers)
    model.eval()
    with torch.inference_mode():
        # Ensure your start prompt token matches your vocabulary bounds
        inp = tok.encode("").unsqueeze(0).to(device)
        generated_output = model.generate(inp, 10)
        print(f"\n[Generated Sample]: {tok.decode(generated_output[0][1:].cpu())}")

    model.train()  # Switch back to training mode for the next epoch loop
    clean_memory_cache()

    print(
        f"Train Epoch: {epoch}, Loss: {loss_meter.calculate():0.4f}, LR: {scheduler.get_last_lr()[0]}"
    )

Training at 0: 100%|██████████| 452/452 [04:53<00:00,  1.54it/s]



[Generated Sample]: <START> : : , , , , , , , ,
Before Clearing, Available memory: 15047.38 MB
Train Epoch: 0, Loss: 8.4752, LR: 9.996145181203615e-05


Training at 1: 100%|██████████| 452/452 [01:22<00:00,  5.50it/s]



[Generated Sample]: <START> : : , , , , , , , ,
Before Clearing, Available memory: 5660.31 MB
Train Epoch: 1, Loss: 7.0254, LR: 9.98458666866564e-05


Training at 2: 100%|██████████| 452/452 [01:18<00:00,  5.73it/s]


...


[Generated Sample]: <START> KING RICHARD III : O , I am I have
Before Clearing, Available memory: 5514.31 MB
Train Epoch: 77, Loss: 4.8019, LR: 1.5413331334360177e-07


Training at 78: 100%|██████████| 452/452 [01:19<00:00,  5.67it/s]



[Generated Sample]: <START> KING RICHARD III : O , I am I have
Before Clearing, Available memory: 5770.31 MB
Train Epoch: 78, Loss: 4.8016, LR: 3.854818796385494e-08


Training at 79: 100%|██████████| 452/452 [05:28<00:00,  1.38it/s]  



[Generated Sample]: <START> KING RICHARD III : O , I am I have
Before Clearing, Available memory: 5678.31 MB
Train Epoch: 79, Loss: 4.8014, LR: 0.0


In [68]:
inp = tok.encode("").unsqueeze(0).to(device)
print(tok.decode(model.generate(inp, 50)[0].cpu()))

<START> KING RICHARD III : O , I am I have you have you not have you have you . I am you , and my lord , you , you , my lord , my heart , my lord , and I have you have you , and my lord


In [69]:
inp = tok.encode("").unsqueeze(0).to(device)
print(tok.decode(model.generate(inp, 200)[0].cpu()))

<START> KING RICHARD III : O , I am I have you have you not have you have you . I am you , and my lord , you , you , my lord , my heart , my lord , and I have you have you , and my lord , And have you , and my lord , I am not , And not not have you , And not , I have you , And not the king , And not your own , and the king . I am not : And not the king . I have you , And not , And , And not , and the king , And , and the king , and the king 's death , and the king 's death , and the king 's son . What , and the king , And , And not , And not you have you do me , And then you , and the king , And , and the king 's the king 's death , And not , and the king , and , and the king , and the king , And , And not , ,


In [70]:
clean_memory_cache()

Before Clearing, Available memory: 2288.92 MB


In [71]:
torch.save(model.state_dict(), "llm.pt")

In [72]:
clean_memory_cache()

Before Clearing, Available memory: 1213.69 MB


In [73]:
clean_memory_cache()

Before Clearing, Available memory: 1213.69 MB


In [74]:
inp = tok.encode("KING").unsqueeze(0).to(device)
generated = tok.decode(model.generate(inp, 200)[0].cpu())

In [108]:
inp = tok.encode("").unsqueeze(0).to(device)
tok.decode(model.generate(inp, 200)[0][1:].cpu())

"KING RICHARD III : O , I am I have you have you not have you have you . I am you , and my lord , you , you , my lord , my heart , my lord , and I have you have you , and my lord , And have you , and my lord , I am not , And not not have you , And not , I have you , And not the king , And not your own , and the king . I am not : And not the king . I have you , And not , And , And not , and the king , And , and the king , and the king 's death , and the king 's death , and the king 's son . What , and the king , And , And not , And not you have you do me , And then you , and the king , And , and the king 's the king 's death , And not , and the king , and , and the king , and the king , And , And not , ,"

In [107]:
print(generated[7:])

 KING RICHARD III : O , I am I have you have you not have you have you . I am you , and my lord , you , you , my lord , my heart , my lord , and I have you have you , and my lord , And have you , and my lord , I am not , And not not have you , And not , I have you , And not the king , And not your own , and the king . I am not : And not the king . I have you , And not , And , And not , and the king , And , and the king , and the king 's death , and the king 's death , and the king 's son . What , and the king , And , And not , And not you have you do me , And then you , and the king , And , and the king 's the king 's death , And not , and the king , and , and the king , and the king , And , And not , , and


In [95]:
x = model.generate(tok.encode("I").unsqueeze(0).to(device), 500).cpu()

In [96]:
len(x)

1

In [97]:
x

tensor([[    0,  1279,    18,  1279,  3176,  9331,    14, 11595,    14, 11595,
            14,  3222,  1279,     8, 12513, 14038,    16,  1279,  7363,  3657,
         14038,    14, 14038,    16,  1279,  3176,  9331,  3595,  2874,  8677,
            14,  1279,  3176,  2874,  8677,    14,  1279,  7363,  2874,  8677,
            14,  3222,  1279,  7363, 14038,    14,  1279,  7363,  3657, 14038,
            14,  1279,  7363,  3657, 14038,    14,  1279,  3176,  9331,  3595,
            14,  3222, 14038,    16,  1279,  7363,  3657, 14038,    14, 14038,
            14,  1279,  7363, 14038,    14,  3222,  9331,    14,  3222,  9157,
          8539,    14,  3361,  1279,  7363,  3657, 14038,    14,  3222,    14,
          3361, 14038,    14,  3222,  7363, 14038,    14,  3361, 14038,    14,
          9157,  8539,    14,  3222, 14038,  3322, 14038,    14,  3222, 14038,
            14,  3222, 14038,  7363, 14038,  7363,  3657, 14038,    14,  1279,
             8, 14038,    14,  3222, 14038,    14,  

In [105]:
tok.decode(x[0][1:])

"I : I am not , sir , sir , and I 'll tell you . I have been you , you . I am not be a man , I am a man , I have a man , and I have you , I have been you , I have been you , I am not be , and you . I have been you , you , I have you , and not , and my lord , as I have been you , and , as you , and have you , as you , my lord , and you are you , and you , and you have you have been you , I 'll you , and you , and my lord , and you , my lord , and say you , and I am not , I have you have been , and have you , and I have you are the king , and you , I 'll tell me , and you , and the world , and , that you , as you , and the king . I have you , and the world , the people , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , an